# The Transformer Architecture

Notes from Jay Alammar's [Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/), based on "Attention is All You Need" (Vaswani et al., 2017).

The baseline model in parameter golf is a transformer — 9 layers, 512 hidden dim, grouped query attention, tied embeddings. Understanding the vanilla architecture first makes the competition's modifications make sense.

## Architecture Overview

Two stacks: an **encoder** (6 layers) and a **decoder** (6 layers).

Each encoder layer has two sub-layers:
1. Self-attention
2. Feed-forward network (applied independently at each position, same weights everywhere)

Each decoder layer has three sub-layers:
1. Masked self-attention (can only see earlier positions)
2. Encoder-decoder attention (queries from decoder, keys/values from encoder output)
3. Feed-forward network

Every sub-layer has a **residual connection** around it followed by **layer normalization**:
```
output = LayerNorm(x + SubLayer(x))
```

The parameter golf baseline is decoder-only (no encoder stack, no encoder-decoder attention). Just masked self-attention and feed-forward, repeated 9 times.

## Self-Attention

Each token's embedding (512 numbers) gets multiplied by three learned weight matrices to produce:
- **Query (Q)** — what this token is looking for
- **Key (K)** — what this token advertises to others
- **Value (V)** — the content this token contributes when attended to

Q, K, V vectors are smaller than the embedding (64 dimensions in the paper).

### How the scores are computed

For each token:
1. Dot product its Q with every token's K → raw attention scores
2. Divide by √d_k (√64 = 8) to keep gradients stable
3. Softmax → attention weights that sum to 1
4. Multiply each token's V by its weight, sum them up → output for this position

In matrix form:

```
Attention(Q, K, V) = softmax(QK^T / √d_k) · V
```

The √d_k scaling matters because without it, large dot products push softmax into regions where gradients vanish.

## Multi-Head Attention

The paper runs 8 attention heads in parallel. Each head has its own Q, K, V weight matrices and produces its own output.

Why multiple heads: different heads can attend to different things. One head might track syntax, another might track coreference. You get multiple representation subspaces instead of one.

The 8 outputs get concatenated and multiplied by a learned W_O matrix to project back to 512 dimensions.

The baseline uses **grouped query attention (GQA)** instead of full multi-head attention — 8 query heads but only 4 key/value heads. Pairs of query heads share the same K and V, cutting KV parameter count in half. Same idea, fewer parameters.

## Positional Encoding

Self-attention is permutation-invariant — shuffling the input tokens produces the same attention scores (just reordered). The model needs position information injected separately.

The paper adds a fixed vector to each embedding before the first encoder layer. These vectors follow sine/cosine functions at different frequencies:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Each dimension gets a different frequency. Low dimensions oscillate fast (nearby positions look different), high dimensions oscillate slowly (capture long-range structure).

The baseline uses **RoPE (Rotary Position Embeddings)** instead. RoPE encodes position by rotating Q and K vectors, so position information enters through the attention dot product rather than being added to embeddings directly.

## The Decoder

The decoder generates tokens one at a time, left to right.

Two differences from the encoder's attention:

1. **Masked self-attention**: future positions are set to -inf before softmax, so token *i* can only attend to tokens 0 through *i*. This enforces autoregressive generation — you can't look ahead.

2. **Encoder-decoder attention**: the decoder creates Q from its own representations but takes K and V from the encoder's final output. This is how the decoder reads the input.

The baseline is decoder-only, so there's no encoder-decoder attention. Just the masked self-attention (called "causal" attention in the code).

## Output Layer

The decoder's top layer output (512 dimensions) passes through:
1. A linear layer that projects to vocabulary size (e.g. 10,000 logits)
2. Softmax to convert logits into a probability distribution

The highest-probability token gets selected.

The baseline uses **tied embeddings** — the input embedding matrix and the output linear layer share the same weights (transposed). One matrix does both jobs, halving the parameter cost of the vocabulary. Leaderboard entries keep these tied embeddings in FP16 (~1MB) even when quantizing everything else harder, because errors in the embedding corrupt both input and output.

## Training

The model trains on (input, target) pairs. At each position, the predicted probability distribution is compared against the actual next token using **cross-entropy loss**.

Label smoothing spreads a small amount of probability to non-target tokens to prevent overconfidence. The baseline does something similar with logit softcapping — squashing logits so they can't exceed ±30.

## Dimensions Reference

| Component | Original Paper | Baseline |
|-----------|---------------|----------|
| Layers | 6 enc + 6 dec | 9 (decoder-only) |
| Hidden dim | 512 | 512 |
| Attention heads | 8 | 8 query, 4 KV (GQA) |
| Head dim | 64 | 64 |
| FFN inner dim | 2048 (4×) | 1024 (2×) |
| Vocab size | ~37K BPE | 1,024 SentencePiece |
| Position encoding | Sinusoidal (fixed) | RoPE (rotation-based) |